# PED 5 · 
## 1. Análisis básico de datos climáticos con ERA5

En esta práctica se trabajará con datos del reanálisis ERA5 descargados por el/la propio/a estudiante desde el portal Copernicus Climate Data Store (CDS).

La tarea es doble:

1. representar una serie temporal diaria de precipitación para un punto próximo a Madrid;
2. elaborar un mapa de precipitación sobre la Comunidad de Madrid a partir de datos en rejilla.

La práctica permite familiarizarse con un flujo de trabajo muy habitual en análisis climático:

- localización y descarga de datos;
- carga del archivo en un entorno de análisis;
- inspección de variables y dimensiones;
- transformación básica de unidades y escalas temporales;
- representación gráfica e interpretación.



In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
csv_file = "AQUI_ESCRIBA_EL_NOMBRE_DE_SU_ARCHIVO.csv"

df = pd.read_csv(csv_file)
df.head()

In [ ]:
df.columns

In [ ]:
# Sustituya el nombre de la columna temporal si fuera necesario
df["valid_time"] = pd.to_datetime(df["valid_time"])

# Cree una columna solo con la fecha
df["date"] = df["valid_time"].dt.date

df.head()

In [ ]:
# Sustituya 'tp' por el nombre real de la columna si fuera distinto
df["tp_mm"] = df["tp"] * 1000

df[["valid_time", "tp", "tp_mm"]].head()

In [ ]:
daily_precip = (
    df.groupby("date", as_index=False)["tp_mm"]
      .sum()
      .rename(columns={"tp_mm": "precip_mm_day"})
)

daily_precip.head()

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(daily_precip["date"], daily_precip["precip_mm_day"])
plt.xlabel("EJE X")
plt.ylabel("EJE Y (UNIDADES)")
plt.title("TITULO")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Mapa de precipitación sobre la Comunidad de Madrid

En este segundo apartado trabajaremos con un archivo NetCDF descargado desde ERA5 en rejilla.

El objetivo es generar un mapa de precipitación para la región de Madrid.
###Acción
Siguiendo las mismas instrucciones que con el archivo anterior, cargue en el entorno de trabajo el archivo de extensión NC.

In [ ]:
nc_file = "AQUI_ESCRIBA_EL_NOMBRE_DE_SU_ARCHIVO.nc"
ds = xr.open_dataset(nc_file)
ds

### Pregunta 3
A partir de la salida del dataset, indique:

- nombre de la variable principal de precipitación;
- dimensiones disponibles;
- coordenadas espaciales incluidas.

In [ ]:
list(ds.data_vars)

In [ ]:
tp = ds["tp"] * 1000  # conversión de m a mm

tp

In [ ]:
# Esta celda asume que el archivo contiene varias horas del mismo día
tp_summer = tp.sum(dim="valid_time")

tp_summer

In [ ]:
tp_madrid = tp_summer.sel(
    latitude=slice(42, 39),
    longitude=slice(-5.0, -2)
)

tp_madrid

In [ ]:

plt.figure(figsize=(7,6))
tp_madrid.plot(cmap="Blues",cbar_kwargs={"label": "Precipitación acumulada (mm)"})
plt.title("TÍTULO GRÁFICA")
plt.xlabel("EJE X")
plt.ylabel("EJE Y")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd

spain = gpd.read_file("spain-communities.geojson")
madrid = spain[spain["name"] == "Madrid"].copy()
madrid = madrid.to_crs("EPSG:4326")

fig, ax = plt.subplots(figsize=(8, 7))

tp_madrid.plot(
    ax=ax,
    cmap="Blues",
    cbar_kwargs={"label": "Precipitación acumulada (mm)"}
)

madrid.boundary.plot(ax=ax, edgecolor="red", linewidth=2)

ax.scatter(-3.70, 40.42, color="yellow", s=40)   # Madrid

ax.set_title("TITULO GRÁFICA")
ax.set_xlabel("EJE X")
ax.set_ylabel("EJE Y")

plt.show()